# 02 — Dialect Identification Training
Fine-tunes WavLM-Large for 63-province Vietnamese dialect classification + regional + ethnic identification.

**Model**: `microsoft/wavlm-large`
**Classes**: 63 provinces + 3 regions + 3 ethnicities
**Dataset**: ViMD (102h) + ViMD-augmented
**Hardware**: Colab T4 GPU


In [ ]:
# ── Setup & Mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/VietSpeech'
DATASET_PATH = f'{PROJECT_ROOT}/hf_dataset'
CHECKPOINT_DIR = f'{PROJECT_ROOT}/checkpoints/dialect'
EXPORT_DIR = f'{PROJECT_ROOT}/weights/dialect'

import sys, os
sys.path.insert(0, PROJECT_ROOT)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Install deps ─────────────────────────────────────────────────────────
!pip install -q transformers datasets accelerate evaluate soundfile
!pip install -q mlflow
print('Deps installed')

In [ ]:
# ── Load config ──────────────────────────────────────────────────────────
import yaml

with open(f'{PROJECT_ROOT}/training/configs/dialect_config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded:')
print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
# ── Load HuggingFace Dataset ─────────────────────────────────────────────
from datasets import load_from_disk, Audio

ds = load_from_disk(DATASET_PATH)
print('Dataset loaded:')
print(ds)

# Cast audio column
ds = ds.cast_column('audio', Audio(sampling_rate=16000))

# Build label encoders
province_list = sorted(ds['train'].unique('province'))
region_list = ['North', 'Central', 'South']
ethnicity_list = ['viet', 'ede', 'bahnar']

province2id = {p: i for i, p in enumerate(province_list)}
region2id = {r: i for i, r in enumerate(region_list)}
ethnicity2id = {e: i for i, e in enumerate(ethnicity_list)}

print(f'Provinces: {len(province2id)} | Regions: {len(region2id)} | Ethnicities: {len(ethnicity2id)}')

In [ ]:
# ── Load WavLM-Large + Feature Extractor ─────────────────────────────────
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import torch.nn as nn

WAVLM_MODEL = cfg.get('model', {}).get('backbone', 'microsoft/wavlm-large')

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_MODEL)
print(f'Feature extractor loaded from {WAVLM_MODEL}')

class DialectClassifier(nn.Module):
    """Multi-task WavLM classifier: province + region + ethnicity."""

    def __init__(self, backbone_name, n_province, n_region, n_ethnicity, dropout=0.1):
        super().__init__()
        self.backbone = WavLMModel.from_pretrained(backbone_name)
        hidden = self.backbone.config.hidden_size  # 1024 for wavlm-large
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.province_head = nn.Linear(hidden, n_province)
        self.region_head = nn.Linear(hidden, n_region)
        self.ethnicity_head = nn.Linear(hidden, n_ethnicity)

    def forward(self, input_values, attention_mask=None):
        out = self.backbone(input_values=input_values, attention_mask=attention_mask)
        # Mean pool over time
        hidden = out.last_hidden_state  # (B, T, H)
        hidden = hidden.mean(dim=1)     # (B, H)
        hidden = self.dropout(hidden)
        return {
            'province': self.province_head(hidden),
            'region': self.region_head(hidden),
            'ethnicity': self.ethnicity_head(hidden),
        }

model = DialectClassifier(
    WAVLM_MODEL,
    n_province=len(province2id),
    n_region=len(region2id),
    n_ethnicity=len(ethnicity2id),
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────
MAX_DURATION_S = cfg.get('data', {}).get('max_duration_s', 10.0)
MAX_SAMPLES = int(MAX_DURATION_S * 16000)

def preprocess(batch):
    audio = [a['array'] for a in batch['audio']]
    inputs = feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors='pt',
        padding='longest',
        max_length=MAX_SAMPLES,
        truncation=True,
    )
    batch['input_values'] = inputs['input_values']
    batch['province_label'] = [province2id[p] for p in batch['province']]
    batch['region_label'] = [region2id[r] for r in batch['region']]
    batch['ethnicity_label'] = [ethnicity2id.get(e, 0) for e in batch['ethnicity']]
    return batch

ds_proc = ds.map(preprocess, batched=True, batch_size=16, num_proc=2)
ds_proc.set_format(type='torch', columns=['input_values', 'province_label', 'region_label', 'ethnicity_label'])
print('Preprocessing done')

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import mlflow

train_cfg = cfg.get('training', {})
BATCH_SIZE  = train_cfg.get('batch_size', 8)
NUM_EPOCHS  = train_cfg.get('num_epochs', 30)
LR          = train_cfg.get('learning_rate', 1e-4)
GRAD_ACC    = train_cfg.get('gradient_accumulation_steps', 4)
WARMUP      = train_cfg.get('warmup_steps', 500)

train_loader = DataLoader(ds_proc['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(ds_proc['val'],   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * len(train_loader))
criterion = nn.CrossEntropyLoss()

mlflow.set_experiment('dialect-identification')

with mlflow.start_run():
    mlflow.log_params({'model': WAVLM_MODEL, 'batch_size': BATCH_SIZE, 'lr': LR, 'epochs': NUM_EPOCHS})

    best_val_acc = 0.0

    for epoch in range(NUM_EPOCHS):
        # ── Training ──
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            inputs = batch['input_values'].to(device)
            prov_labels = batch['province_label'].to(device)
            reg_labels = batch['region_label'].to(device)
            eth_labels = batch['ethnicity_label'].to(device)

            logits = model(inputs)
            loss = (
                criterion(logits['province'], prov_labels) * 0.6 +
                criterion(logits['region'], reg_labels) * 0.3 +
                criterion(logits['ethnicity'], eth_labels) * 0.1
            ) / GRAD_ACC

            loss.backward()
            total_loss += loss.item() * GRAD_ACC

            if (step + 1) % GRAD_ACC == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        avg_train_loss = total_loss / len(train_loader)

        # ── Validation ──
        model.eval()
        correct_prov, total = 0, 0

        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['input_values'].to(device)
                prov_labels = batch['province_label'].to(device)
                logits = model(inputs)
                preds = logits['province'].argmax(dim=-1)
                correct_prov += (preds == prov_labels).sum().item()
                total += prov_labels.size(0)

        val_acc = correct_prov / total
        mlflow.log_metrics({'train_loss': avg_train_loss, 'val_province_acc': val_acc}, step=epoch)
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}: loss={avg_train_loss:.4f}, val_prov_acc={val_acc:.3f}')

        # ── Save best checkpoint ──
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'{CHECKPOINT_DIR}/best_dialect.pt')
            print(f'  → New best checkpoint saved (acc={val_acc:.3f})')

    print(f'\nTraining complete. Best val province accuracy: {best_val_acc:.3f}')
    mlflow.log_metric('best_val_province_acc', best_val_acc)

In [ ]:
# ── Export model ──────────────────────────────────────────────────────────
import json

# Load best checkpoint
model.load_state_dict(torch.load(f'{CHECKPOINT_DIR}/best_dialect.pt', map_location=device))
model.eval()

# Save full model + label maps
os.makedirs(EXPORT_DIR, exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'province2id': province2id,
    'region2id': region2id,
    'ethnicity2id': ethnicity2id,
    'backbone': WAVLM_MODEL,
    'best_val_acc': best_val_acc,
}, f'{EXPORT_DIR}/dialect_model.pt')

# Also save label maps as JSON
with open(f'{EXPORT_DIR}/label_maps.json', 'w') as f:
    json.dump({'province2id': province2id, 'region2id': region2id, 'ethnicity2id': ethnicity2id}, f, indent=2, ensure_ascii=False)

print(f'Model exported to {EXPORT_DIR}/')
print(f'Files: {os.listdir(EXPORT_DIR)}')

In [ ]:
# ── Download checkpoint locally ───────────────────────────────────────────
# Option A: Download via Google Colab file download (max ~2GB)
from google.colab import files
# files.download(f'{EXPORT_DIR}/dialect_model.pt')  # Uncomment to download

# Option B: File is already saved in Google Drive at EXPORT_DIR
print(f'Model saved to Google Drive at: {EXPORT_DIR}')
print('You can copy it locally by running:')
print(f'  cp "{EXPORT_DIR}/dialect_model.pt" ./storage/weights/dialect/')

## ✓ Dialect Identification Training Complete
**Model saved to**: `{EXPORT_DIR}/dialect_model.pt`

**To use locally after training**:
1. The model is saved in your Google Drive at `VietSpeech/weights/dialect/`
2. Copy `dialect_model.pt` and `label_maps.json` to `storage/weights/dialect/`
3. `DialectClassifier.load()` in `models/dialect/dialect_classifier.py` will pick it up automatically

**Next**: Run `03_asr_whisper.ipynb`